## Fase 3 — Pronóstico de los productos piloto

Este notebook toma **solo los productos piloto** de la clasificación de la Fase 2 (Actividades 3.1 a 3.4 y 4.2) y hace tres cosas:

1. **Analiza sus series de tiempo**: cómo se ven, si tienen tendencia, si tienen estacionalidad y qué tan variables son.
2. **Los pronostica** con los métodos referenciados en el trabajo y los compara con métodos simples:

| Familia | Métodos | Referencia |
|---|---|---|
| Simples (referencia) | Ingenuo, estacional ingenuo, promedio móvil 3, suavizado exponencial | Hyndman y Athanasopoulos (2021) |
| Estadísticos | ETS y SARIMA | Hyndman y Athanasopoulos (2021) |
| Aprendizaje automático | XGBoost y Random Forest | Chen y Guestrin (2016); Breiman (2001) |
| Aprendizaje profundo | LSTM | Chidiebube et al. (2025); Hewamalage et al. (2021) |
| Híbrido | SARIMA + CNN-LSTM sobre los residuos | Huang y Zhou (2025); Zhang (2003) |

3. **Entrega el modelo propuesto**: lo entrena con toda la historia, guarda sus archivos y pronostica los próximos tres meses (sección 7).

Croston-SBA y TSB no se incluyen: son para demanda intermitente, y los productos piloto tienen pedidos todos los meses (con demanda continua, Croston equivale a un suavizado exponencial).

**Cómo se evalúa.** Se usa validación de origen móvil: para cada uno de los últimos 9 meses se entrena con lo ocurrido antes y se pronostica ese mes, a 1, 2 y 3 meses de distancia. Así cada pronóstico se hace solo con información que se tenía en ese momento.

**Métricas.** *MAE*: error promedio en unidades. *WMAPE*: error total sobre unidades totales (0,20 = se equivoca en 20 % de lo pedido). *MASE*: error del método dividido por el error medio de repetir el mes anterior en los meses de entrenamiento; menor a 1 significa menos error que ese método ingenuo en la historia.

**Paso previo: TensorFlow.** El LSTM y el híbrido necesitan TensorFlow, que no viene en el entorno estándar de Fabric. La celda siguiente lo instala en la sesión **solo si falta** (tarda uno o dos minutos la primera vez). Se fija la versión 2.15.1 porque es compatible con la versión de NumPy de Fabric y no cambia las demás librerías.

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("tensorflow") is None:
    print("Instalando TensorFlow...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tensorflow-cpu==2.15.1"])
    print("TensorFlow instalado.")
else:
    print("TensorFlow ya está disponible.")

In [ ]:
import os; os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"     # silencia los mensajes de TensorFlow
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from IPython.display import display
from scipy import stats
from statsmodels.tsa.seasonal import STL
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX
import time
try:
    import tensorflow as tf
    tf.get_logger().setLevel("ERROR")
    HAY_TF = True
except ImportError:
    HAY_TF = False
    print("ATENCIÓN: TensorFlow no está disponible; se omitirán LSTM y el híbrido. Revise la celda de instalación anterior.")

# Fabric registra automáticamente cada entrenamiento en MLflow y eso lo vuelve muy lento; se desactiva.
try:
    import mlflow; mlflow.autolog(disable=True)
except Exception:
    pass

# ----------------------------- PARÁMETROS -----------------------------
TABLA_CLS = "fs2_clasificacion_linea"     # clasificación de la Fase 2
TABLA_PILOTO = "fs2_abt_sku_mes_piloto"   # historia mensual del piloto preparada en la Fase 2 (A2.4)
TABLA_MES = "gld_abt_sku_mes"             # ABT mensual: solo para el Random Forest global (toda la línea)
COL_Y = "unidades_pedidas"                # variable a pronosticar
HORIZONTES = (1, 2, 3)                    # meses hacia adelante
N_PRUEBA = 9                              # últimos meses usados como prueba
LAGS = (1, 2, 3, 6)                       # meses anteriores usados como variables
SEED = 42

PARAMS_XGB = dict(n_estimators=200, max_depth=2, learning_rate=0.05, subsample=0.8,
                  colsample_bytree=0.8, random_state=SEED, n_jobs=1, verbosity=0)
PARAMS_RF = dict(n_estimators=300, max_depth=4, min_samples_leaf=2, random_state=SEED, n_jobs=1)

VENTANA = 6                               # meses de entrada de las redes LSTM
EPOCAS = 15                               # épocas de entrenamiento de las redes...
REPETICIONES = 10                         # ...con los datos repetidos 10 veces por época (equivale a 150 pasadas, más rápido)

COLORES = {"Real": "#1d2d3a", "Mejor método simple": "#9ca3af", "XGBoost": "#eb6834", "Random Forest": "#2a78d6",
           "ETS": "#1baf7a", "SARIMA": "#7c3aed", "LSTM": "#d97706", "Híbrido ARIMA-CNN-LSTM": "#be185d"}

# Figuras para el documento: cada gráfica que se muestra con plt.show() se guarda también como PNG a 300 dpi
# en el Lakehouse (Files/figuras_tesis/fase3/figura_NN.png, en el orden en que aparecen en el notebook).
# No cambia ningún cálculo; solo deja una copia de cada figura con resolución suficiente para el documento.
import os
plt.rcParams["savefig.dpi"] = 300
DIR_FIGURAS = "/lakehouse/default/Files/figuras_tesis/fase3"
if not getattr(plt.show, "_guarda_figuras", False):
    _show_original = plt.show
    _figuras_guardadas = [0]

    def _show_y_guardar(*args, **kwargs):
        if os.path.isdir("/lakehouse/default/Files"):
            os.makedirs(DIR_FIGURAS, exist_ok=True)
            for num in plt.get_fignums():
                _figuras_guardadas[0] += 1
                plt.figure(num).savefig(f"{DIR_FIGURAS}/figura_{_figuras_guardadas[0]:02d}.png",
                                        bbox_inches="tight", facecolor="white")
        return _show_original(*args, **kwargs)

    _show_y_guardar._guarda_figuras = True
    plt.show = _show_y_guardar


### 1. Datos de los productos piloto

Se leen los productos con `piloto = True` en la clasificación de la Fase 2 y su serie mensual de unidades pedidas, tomada de `fs2_abt_sku_mes_piloto`, el dataset del piloto que preparó la Fase 2 (Actividad 2.4). Los productos se numeran del 1 al 5 igual que en la Tabla 16 de la tesis. Los meses sin registro se completan con cero y se quita el mes de lanzamiento (suele ser un mes incompleto).

In [ ]:
cls = spark.table(TABLA_CLS).toPandas()
cls["ItemCode"] = cls["ItemCode"].astype(str)
piloto = cls[cls["piloto"] == True].copy()

# Confidencialidad: los productos se muestran como "Producto 1" a "Producto 5", con la misma numeración de la
# tesis (Tabla 16: de más a menos votos del consenso y, en empate, por código). Los códigos y nombres reales no
# aparecen en ninguna salida; la equivalencia solo se guarda en el Lakehouse (tabla fs3_piloto_alias).
if "votos_piloto" in piloto.columns:
    orden = piloto.sort_values(["votos_piloto", "ItemCode"], ascending=[False, True])["ItemCode"]
else:
    print("Aviso: la clasificación no trae votos_piloto; los productos se numeran por código.")
    orden = sorted(piloto["ItemCode"])
ALIAS = {k: f"Producto {i + 1}" for i, k in enumerate(orden)}
piloto["ItemCode"] = piloto["ItemCode"].map(ALIAS)
piloto = piloto.sort_values("ItemCode")
PRODUCTOS = list(piloto["ItemCode"])

mes = spark.table(TABLA_PILOTO).toPandas()          # dataset del piloto que dejó la Fase 2 (A2.4)
mes["ItemCode"] = mes["ItemCode"].astype(str)
mes = mes[mes["ItemCode"].isin(ALIAS)].copy()
mes["ItemCode"] = mes["ItemCode"].map(ALIAS)
mes["mes"] = pd.to_datetime(mes["mes"]).dt.to_period("M").dt.to_timestamp()
COMERCIALES = [c for c in ["mes_promocional", "descuento_incremental"] if c in mes.columns]
EXTRA = [c for c in ["mes_alza_precio", "mes_previo_alza", "num_pedidos", "num_clientes"] if c in mes.columns]   # para el análisis de picos

def preparar_serie(d):
    d = d.sort_values("mes").drop_duplicates("mes").set_index("mes")
    if "mes_lanzamiento" in d.columns:
        d = d[~d["mes_lanzamiento"].fillna(False).astype(bool)]
    d = d.reindex(pd.date_range(d.index.min(), d.index.max(), freq="MS"))
    d[COL_Y] = d[COL_Y].fillna(0).clip(lower=0)
    for c in COMERCIALES + EXTRA + ["pct_dias_quiebre"]:
        d[c] = d[c].astype(float).fillna(0) if c in d.columns else 0.0
    return d[[COL_Y, "pct_dias_quiebre"] + COMERCIALES + EXTRA]

SERIES = {k: preparar_serie(g) for k, g in mes.groupby("ItemCode")}

print(f"Productos piloto: {len(PRODUCTOS)} | filas leídas de {TABLA_PILOTO}: {len(mes)} | meses usados: {sum(len(d) for d in SERIES.values())} (sin el mes de lanzamiento)")
display(piloto[["ItemCode", "P_consenso", "F_consenso"]].rename(columns={"ItemCode": "Producto"}).reset_index(drop=True))

### 2. Análisis de las series de tiempo

#### 2.1 Cómo se ve cada serie

Cada gráfica muestra las unidades pedidas por mes (línea oscura), el promedio móvil de 6 meses (línea gris, para ver la tendencia sin el ruido) y los meses con más de 10 % de días en quiebre de inventario (✕ roja). En esos meses lo pedido puede ser menor que la demanda real porque no había producto. La franja amarilla es el periodo de prueba de los pronósticos.

In [ ]:
fig, axes = plt.subplots(len(PRODUCTOS), 1, figsize=(12, 2.8 * len(PRODUCTOS)), squeeze=False)
for ax, k in zip(axes[:, 0], PRODUCTOS):
    d = SERIES[k]; y = d[COL_Y]
    ax.axvspan(d.index[-N_PRUEBA], d.index[-1], color="#e9c46a", alpha=.25, lw=0)
    ax.plot(d.index, y, color=COLORES["Real"], lw=1.8, marker="o", ms=3, label="Unidades pedidas")
    ax.plot(d.index, y.rolling(6, min_periods=3).mean(), color="#9ca3af", lw=2, label="Promedio móvil 6 meses")
    q = d["pct_dias_quiebre"] > 0.10
    ax.scatter(d.index[q], y[q], marker="x", color="#c0392b", s=40, zorder=5, label="Mes con quiebre")
    ax.set_title(f"{k}", loc="left", fontsize=10)
    ax.set_ylim(bottom=0); ax.grid(alpha=.2)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b-%y"))
axes[0, 0].legend(fontsize=8, frameon=False, loc="upper left", bbox_to_anchor=(1.01, 1))
plt.tight_layout(); plt.show()

#### 2.2 Descomposición: tendencia, estacionalidad y resto

La descomposición STL separa cada serie en tres partes: la **tendencia** (hacia dónde va el nivel), la **estacionalidad** (el patrón que se repite cada 12 meses) y el **resto** (lo que no se explica, el ruido). Si la parte estacional tiene una amplitud grande comparada con el resto, el mes del año ayuda a pronosticar.

In [ ]:
for k in PRODUCTOS:
    y = SERIES[k][COL_Y]
    if len(y) < 24:
        print(f"{k}: menos de 24 meses, no se puede descomponer"); continue
    st = STL(y, period=12, robust=True).fit()
    fig, axes = plt.subplots(1, 3, figsize=(14, 2.6))
    for ax, serie, tit in zip(axes, [st.trend, st.seasonal, st.resid], ["Tendencia", "Estacionalidad", "Resto"]):
        ax.plot(serie.index, serie, color="#2a78d6", lw=1.6)
        ax.set_title(f"{k} · {tit}", loc="left", fontsize=9); ax.grid(alpha=.2)
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%b-%y")); ax.tick_params(labelsize=7)
    axes[2].axhline(0, color="#9ca3af", lw=.8)
    plt.tight_layout(); plt.show()

#### 2.3 Resumen de las series

| Indicador | Qué mide |
|---|---|
| `meses` | Largo de la serie. |
| `promedio_mes` | Unidades pedidas promedio por mes. |
| `pct_meses_con_pedido` | Proporción de meses con algún pedido (1 = pide todos los meses). |
| `CV` | Desviación estándar / promedio. Menor a 0,5 es una serie estable; mayor a 1, muy variable. |
| `tendencia` | Prueba de Mann-Kendall: *creciente* o *decreciente* si es significativa (p < 0,05). |
| `cambio_anual_pct` | Cuánto cambia el nivel en un año según la pendiente de la serie (Theil-Sen), en % del promedio. |
| `fuerza_estacional` | De 0 a 1, qué tanto pesa la estacionalidad frente al ruido (descomposición STL). Mayor a 0,6 es fuerte; entre 0,3 y 0,6, moderada. |
| `autocorrelacion_1` | Qué tanto se parece un mes al anterior. Cerca de 1, la serie tiene "memoria" y los meses anteriores ayudan a pronosticar. |
| `pct_meses_quiebre` | Proporción de meses con más de 10 % de días sin inventario. |

In [ ]:
def resumen_serie(k):
    d = SERIES[k]; y = d[COL_Y].to_numpy(float); t = np.arange(len(y))
    tau, p = stats.kendalltau(t, y)
    pendiente = stats.theilslopes(y, t)[0]
    r = {"ItemCode": k, "meses": len(y), "promedio_mes": y.mean(), "pct_meses_con_pedido": (y > 0).mean(),
         "CV": y.std() / y.mean() if y.mean() > 0 else np.nan,
         "tendencia": ("creciente" if tau > 0 else "decreciente") if p < 0.05 else "sin tendencia",
         "cambio_anual_pct": 100 * pendiente * 12 / y.mean() if y.mean() > 0 else np.nan,
         "fuerza_estacional": np.nan,
         "autocorrelacion_1": pd.Series(y).autocorr(1),
         "pct_meses_quiebre": (d["pct_dias_quiebre"] > 0.10).mean()}
    if len(y) >= 24:
        st = STL(d[COL_Y], period=12, robust=True).fit()
        r["fuerza_estacional"] = max(0.0, 1 - st.resid.var() / (st.seasonal + st.resid).var())
    return r

RESUMEN_SERIES = pd.DataFrame([resumen_serie(k) for k in PRODUCTOS])
display(RESUMEN_SERIES.rename(columns={"ItemCode": "Producto"}).round(2))

**Cómo leer la tabla.** La tabla resume cada serie: su largo, el promedio mensual, la variabilidad (coeficiente de variación), la tendencia anual y su significancia (Mann-Kendall y pendiente de Theil-Sen), la fuerza de la estacionalidad, la autocorrelación con el mes anterior y la proporción de meses con quiebre de inventario.

Estos indicadores orientan qué métodos tienen sentido. Si todas las series tienen pedidos en todos los meses, la demanda es continua y no se usan métodos para demanda intermitente (Croston-SBA o TSB). Una autocorrelación baja indica que el mes anterior dice poco sobre el siguiente, lo que limita a los métodos que se apoyan en rezagos. Una proporción alta de meses con quiebre indica que, en esos meses, las unidades pedidas pueden subestimar la demanda real. Con pocos años de historia, la estacionalidad se estima con pocos ciclos y debe leerse con cautela. Los resultados obtenidos con los datos de la empresa y su interpretación se presentan en el documento de la tesis.

#### 2.4 Análisis de los picos

Un **pico** es un mes en que la demanda queda muy por encima de lo habitual. Para detectarlo se compara cada mes con la mediana móvil centrada de 5 meses, el mismo criterio de la Fase 1: si las unidades pedidas son al menos **1,5 veces** esa mediana, el mes se marca como pico. Se usa la mediana y no el promedio porque el propio pico no la mueve.

Para cada pico se revisan las causas posibles que hay en la tabla base analítica:

| Posible causa | Cómo se identifica | Por qué genera un pico |
|---|---|---|
| Promoción | Mes promocional (descuento incremental ≥ 3 puntos) | El descuento adelanta o aumenta las compras |
| Compra anticipada | El mes siguiente hay alza de precio | Los clientes se abastecen antes de que suba el precio |
| Rebote tras quiebre | El mes anterior tuvo más de 10 % de días sin inventario | Se acumulan pedidos que no se pudieron atender |
| Pedido concentrado | Unidades por pedido al menos 2 veces la mediana del producto | Un cliente hace un pedido excepcionalmente grande |

Un pico puede tener varias causas o ninguna identificable con los datos. También se marca si al mes siguiente la demanda cae por debajo del 80 % de lo habitual: es la señal de que el pico **adelantó** compras en vez de crear demanda nueva, algo que un modelo de pronóstico debería tener en cuenta.

In [ ]:
UMBRAL_PICO = 1.5
filas, meses = [], []
for k in PRODUCTOS:
    d = SERIES[k].copy(); y = d[COL_Y]
    d["veces"] = y / y.rolling(5, center=True, min_periods=3).median().replace(0, np.nan)
    if "num_pedidos" in d.columns:
        unid_pedido = y / d["num_pedidos"].replace(0, np.nan)
        d["pedido_rel"] = unid_pedido / unid_pedido.median()
    d["quiebre_previo"] = d["pct_dias_quiebre"].shift(1) > 0.10
    d["Producto"] = k
    meses.append(d)
    for j, (fecha, r) in enumerate(d.iterrows()):
        if not r["veces"] >= UMBRAL_PICO: continue
        causas = []
        if r.get("mes_promocional", 0) > 0: causas.append("promoción")
        if r.get("mes_previo_alza", 0) > 0: causas.append("compra anticipada")
        if r["quiebre_previo"]: causas.append("rebote tras quiebre")
        if r.get("pedido_rel", 0) >= 2: causas.append("pedido concentrado")
        siguiente = d["veces"].iloc[j + 1] if j + 1 < len(d) else np.nan
        filas.append({"Producto": k, "Mes": f"{fecha:%Y-%m}", "Unidades": int(r[COL_Y]), "Veces lo habitual": round(r["veces"], 2),
                      "Descuento incremental (pp)": round(float(r.get("descuento_incremental", 0)), 1),
                      "Posibles causas": ", ".join(causas) or "sin causa identificada",
                      "Cae al mes siguiente": "sí" if siguiente < 0.8 else "no"})
PICOS = pd.DataFrame(filas)
MESES = pd.concat(meses)

print(f"Picos detectados: {len(PICOS)} en {PICOS['Producto'].nunique() if len(PICOS) else 0} productos")
display(PICOS)
if len(PICOS):
    resumen_causas = PICOS["Posibles causas"].str.split(", ").explode().value_counts().rename("Picos").to_frame()
    resumen_causas["% de los picos"] = (resumen_causas["Picos"] / len(PICOS)).round(2)
    display(resumen_causas)
    print(f"Picos seguidos de una caída (< 80 % de lo habitual): {(PICOS['Cae al mes siguiente'] == 'sí').sum()} de {len(PICOS)}")

**¿Cada causa sube la demanda en general?** La tabla anterior solo mira los picos. La siguiente compara **todos los meses** de los cinco productos: cuántas veces lo habitual se vende en los meses con cada condición frente a los meses sin ella. La prueba de Mann-Whitney dice si la diferencia es estadísticamente significativa (p < 0,05).

In [ ]:
from scipy.stats import mannwhitneyu
condiciones = {"Promoción": MESES.get("mes_promocional", pd.Series(0, index=MESES.index)) > 0,
               "Mes previo a alza de precio": MESES.get("mes_previo_alza", pd.Series(0, index=MESES.index)) > 0,
               "Mes de alza de precio": MESES.get("mes_alza_precio", pd.Series(0, index=MESES.index)) > 0,
               "Después de un quiebre": MESES["quiebre_previo"]}
filas = []
v = MESES["veces"]
for nom, cond in condiciones.items():
    con, sin = v[cond.to_numpy()].dropna(), v[~cond.to_numpy()].dropna()
    if len(con) < 3:
        filas.append({"Condición": nom, "Meses con la condición": len(con)}); continue
    filas.append({"Condición": nom, "Meses con la condición": len(con), "Veces lo habitual (con)": round(con.median(), 2),
                  "Veces lo habitual (sin)": round(sin.median(), 2), "p (Mann-Whitney)": round(mannwhitneyu(con, sin).pvalue, 3)})
display(pd.DataFrame(filas))

In [ ]:
fig, axes = plt.subplots(len(PRODUCTOS), 1, figsize=(13, 2.8 * len(PRODUCTOS)), squeeze=False)
for ax, k in zip(axes[:, 0], PRODUCTOS):
    d = MESES[MESES["Producto"] == k]; y = d[COL_Y]
    if "mes_promocional" in d.columns:
        for f in d.index[d["mes_promocional"] > 0]:
            ax.axvspan(f - pd.Timedelta(days=14), f + pd.Timedelta(days=14), color="#1baf7a", alpha=.15, lw=0)
    if "mes_previo_alza" in d.columns:
        for f in d.index[d["mes_previo_alza"] > 0]:
            ax.axvline(f, color="#7c3aed", ls="--", lw=1)
    ax.plot(d.index, y, color=COLORES["Real"], lw=1.6, marker="o", ms=3)
    ax.plot(d.index, y.rolling(5, center=True, min_periods=3).median(), color="#9ca3af", lw=1.5)
    p = d["veces"] >= UMBRAL_PICO
    ax.scatter(d.index[p], y[p], marker="^", s=70, color="#eb6834", zorder=5)
    ax.set_title(k, loc="left", fontsize=10); ax.set_ylim(bottom=0); ax.grid(alpha=.2)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b-%y"))
leyenda = [Line2D([], [], color=COLORES["Real"], marker="o", ms=3, label="Unidades pedidas"),
           Line2D([], [], color="#9ca3af", lw=1.5, label="Mediana móvil de 5 meses"),
           Line2D([], [], color="#eb6834", marker="^", ms=8, lw=0, label=f"Pico (≥ {UMBRAL_PICO} veces la mediana)"),
           Patch(color="#1baf7a", alpha=.3, label="Mes promocional"),
           Line2D([], [], color="#7c3aed", ls="--", label="Mes previo a alza de precio")]
axes[0, 0].legend(handles=leyenda, fontsize=8, frameon=False, loc="upper left", bbox_to_anchor=(1.01, 1))
plt.tight_layout(); plt.show()

**Cómo leer el análisis de picos.** Cada pico se asocia con la causa más probable que muestran los datos: demanda represada después de un mes con quiebre, promoción, pedido concentrado en pocos clientes o compra anticipada antes de un alza de precio. Los picos sin causa identificable deben validarse con el área comercial. Conviene revisar además:

1. **Picos simultáneos.** Si varios productos tienen picos el mismo mes, es probable que se deban a un evento comercial de la línea (un pedido grande o una campaña) más que al comportamiento de cada producto.
2. **Momento.** Si los picos se concentran antes del periodo de prueba, los errores de la sección 4 se explican más por cambios de nivel que por picos.
3. **Caída posterior.** Un pico seguido de una caída al mes siguiente sugiere compras adelantadas, no demanda nueva.

La prueba de Mann-Whitney indica si cada condición (promoción, alza de precio, quiebre previo) sube la demanda de forma significativa en el conjunto de todos los meses. Los resultados obtenidos con los datos de la empresa y su interpretación se presentan en el documento de la tesis.

**Decisión: los modelos se entrenan con las series originales, sin limpiar los picos.** Se probó una limpieza simple, que reemplazaba los picos sin causa por la mediana de los cinco meses anteriores, y empeoró el pronóstico: la regla confunde los cambios de nivel con picos y deja la serie anclada en el nivel anterior. Además, los picos que siguen a un quiebre son demanda que sí ocurrió, por lo que retirarlos quitaría información real.

### 3. Pronósticos

#### 3.1 Variables del modelo

Para pronosticar el mes *t + h* con lo que se sabe en el mes *t*, cada fila del modelo tiene:

- las unidades de los meses anteriores (rezagos 1, 2, 3 y 6),
- el promedio de los últimos 3 y 6 meses,
- el % de días en quiebre del último mes,
- el mes del año que se pronostica (en seno y coseno, para que diciembre quede cerca de enero),
- las variables comerciales planificadas del mes pronosticado (promoción y descuento), suponiendo que la empresa ya las conoce al pronosticar.

Se entrena **un modelo por producto y por horizonte**.

In [ ]:
def construir_tabla(d, h):
    y = d[COL_Y].to_numpy(float); q = d["pct_dias_quiebre"].to_numpy(float); fechas = d.index
    filas = []
    for t in range(6, len(y) - h + 1):          # t = meses observados al momento de pronosticar
        j = t + h - 1                            # posición del mes que se pronostica
        f = {"fecha": fechas[j], "fecha_origen": fechas[t - 1], "y": y[j]}
        for L in LAGS: f[f"lag{L}"] = y[t - L]
        f["mm3"], f["mm6"] = y[t - 3:t].mean(), y[t - 6:t].mean()
        f["quiebre_ultimo_mes"] = q[t - 1]
        f["mes_sin"], f["mes_cos"] = np.sin(2 * np.pi * fechas[j].month / 12), np.cos(2 * np.pi * fechas[j].month / 12)
        for c in COMERCIALES: f[c] = d[c].iloc[j]
        filas.append(f)
    return pd.DataFrame(filas)

VARIABLES = [f"lag{L}" for L in LAGS] + ["mm3", "mm6", "quiebre_ultimo_mes", "mes_sin", "mes_cos"] + COMERCIALES

#### 3.2 Métodos simples de referencia

Un modelo de aprendizaje automático solo vale la pena si le gana a métodos simples. Se comparan cuatro:

- **Ingenuo**: el próximo mes será igual al último observado.
- **Estacional ingenuo**: será igual al mismo mes del año anterior.
- **Promedio móvil 3**: el promedio de los últimos 3 meses.
- **Suavizado exponencial**: un promedio que da más peso a los meses recientes (α = 0,3).

In [ ]:
def suavizado(y, a=0.3):
    nivel = y[0]
    for v in y[1:]: nivel = a * v + (1 - a) * nivel
    return nivel

def metodos_simples(y, j, h):
    obs = y[:j - h + 1]                          # lo observado al momento de pronosticar
    return {"Ingenuo": obs[-1],
            "Estacional ingenuo": y[j - 12] if j - 12 < len(obs) and j >= 12 else obs[-1],
            "Promedio móvil 3": obs[-3:].mean(),
            "Suavizado exponencial": suavizado(obs)}

#### 3.3 Validación de origen móvil: XGBoost y Random Forest

Para cada producto, horizonte y mes de prueba: se entrena XGBoost y Random Forest **solo con las filas cuyo mes objetivo ya había ocurrido** en el momento del pronóstico, y se pronostica el mes de prueba. Los métodos simples se calculan con la misma información.

In [ ]:
filas = []
for k in PRODUCTOS:
    d = SERIES[k]; y = d[COL_Y].to_numpy(float)
    meses_prueba = d.index[-N_PRUEBA:]
    for h in HORIZONTES:
        tabla = construir_tabla(d, h)
        for fecha in meses_prueba:
            prueba = tabla[tabla["fecha"] == fecha]
            if prueba.empty: continue
            entreno = tabla[tabla["fecha"] <= prueba["fecha_origen"].iloc[0]]
            real = prueba["y"].iloc[0]
            for nom, modelo in [("XGBoost", XGBRegressor(**PARAMS_XGB)), ("Random Forest", RandomForestRegressor(**PARAMS_RF))]:
                modelo.fit(entreno[VARIABLES], entreno["y"])
                pred = max(float(modelo.predict(prueba[VARIABLES])[0]), 0)
                filas.append(dict(ItemCode=k, h=h, fecha=fecha, real=real, pred=pred, metodo=nom))
            j = d.index.get_loc(fecha)
            for nom, pred in metodos_simples(y, j, h).items():
                filas.append(dict(ItemCode=k, h=h, fecha=fecha, real=real, pred=float(pred), metodo=nom))

PRED = pd.DataFrame(filas)
print(f"Pronósticos generados: {len(PRED):,}")

#### 3.4 Métodos estadísticos: ETS y SARIMA

A diferencia de XGBoost y Random Forest, estos modelos se ajustan sobre la serie misma y pronostican varios meses hacia adelante desde un mismo origen.

- **ETS (Holt-Winters):** suavizado exponencial con nivel y, si mejora el ajuste, tendencia (normal o amortiguada) y estacionalidad aditiva de 12 meses.
- **SARIMA:** modelo autorregresivo de medias móviles. Se prueban órdenes p y q de 0 a 2, con o sin diferenciación (d = 0 o 1) y con o sin un término autorregresivo estacional de 12 meses.

**Cómo se elige la configuración.** Para cada producto se elige una sola vez, con los datos anteriores al primer origen de prueba, la configuración de menor AICc (criterio de información que penaliza la complejidad). En cada origen se reestiman los coeficientes con la historia disponible y se pronostican los siguientes 1, 2 y 3 meses.

In [ ]:
H_MAX = max(HORIZONTES)

def ets_ajustar(obs, cfg):
    tendencia, amortiguada, estacional = cfg
    return ExponentialSmoothing(obs, trend=tendencia, damped_trend=amortiguada, seasonal=estacional,
                                seasonal_periods=12 if estacional else None, initialization_method="estimated").fit()

def ets_elegir(obs):
    candidatos = [(None, False, None), ("add", False, None), ("add", True, None)]
    if len(obs) >= 24:
        candidatos += [(None, False, "add"), ("add", True, "add")]
    aicc = {}
    for c in candidatos:
        try: aicc[c] = ets_ajustar(obs, c).aicc
        except Exception: pass
    aicc = {c: v for c, v in aicc.items() if np.isfinite(v)}
    return min(aicc, key=aicc.get)

def sarima_ajustar(obs, orden, estacional):
    return SARIMAX(obs, order=orden, seasonal_order=estacional, trend="c" if orden[1] == 0 else "n").fit(disp=False)

def sarima_elegir(obs):
    estacionales = [(0, 0, 0, 0)] + ([(1, 0, 0, 12)] if len(obs) >= 24 else [])
    aicc = {}
    for p in range(3):
        for d in range(2):
            for q in range(3):
                for est in estacionales:
                    try: aicc[((p, d, q), est)] = sarima_ajustar(obs, (p, d, q), est).aicc
                    except Exception: pass
    aicc = {c: v for c, v in aicc.items() if np.isfinite(v)}
    return min(aicc, key=aicc.get)

def agregar(filas, k, L, metodo, pron):
    # pron = pronóstico de 1, 2 y 3 meses hecho con los primeros L meses; se guardan los que caen en la prueba
    d = SERIES[k]; y = d[COL_Y].to_numpy(float); n = len(y)
    for h in HORIZONTES:
        j = L + h - 1
        if n - N_PRUEBA <= j < n:
            filas.append(dict(ItemCode=k, h=h, fecha=d.index[j], real=y[j], pred=max(float(pron[h - 1]), 0.0), metodo=metodo))

warnings.simplefilter("ignore")                     # statsmodels avisa cuando un ajuste converge con dificultad
t0 = time.time()
filas_est, CONFIG, ARIMA_FIT = [], [], {}
for k in PRODUCTOS:
    y = SERIES[k][COL_Y].to_numpy(float); n = len(y)
    L0 = n - N_PRUEBA - H_MAX + 1                       # historia disponible en el primer origen de prueba
    cfg_ets = ets_elegir(y[:L0])
    orden, est = sarima_elegir(y[:L0])
    CONFIG.append({"ItemCode": k,
                   "ETS": f"tendencia {'amortiguada' if cfg_ets[1] else ('aditiva' if cfg_ets[0] else 'no')}, estacionalidad {'aditiva' if cfg_ets[2] else 'no'}",
                   "SARIMA (p,d,q)x(P,D,Q,s)": f"{orden}x{est}"})
    for L in range(L0, n):
        obs = y[:L]
        try:
            agregar(filas_est, k, L, "ETS", ets_ajustar(obs, cfg_ets).forecast(H_MAX))
        except Exception as e:
            print(f"ETS {k}, origen {L}: {e}")
        try:
            fit = sarima_ajustar(obs, orden, est)
            pron = np.asarray(fit.forecast(H_MAX), float)
            agregar(filas_est, k, L, "SARIMA", pron)
            ARIMA_FIT[(k, L)] = (pron, np.asarray(fit.resid, float))   # se reutiliza en el híbrido
        except Exception as e:
            print(f"SARIMA {k}, origen {L}: {e}")

PRED = pd.concat([PRED, pd.DataFrame(filas_est)], ignore_index=True)
print(f"ETS y SARIMA: {len(filas_est):,} pronósticos en {time.time() - t0:.0f} s")
print("Configuración elegida por producto (menor AICc):")
display(pd.DataFrame(CONFIG).rename(columns={"ItemCode": "Producto"}))

**Cómo leer la tabla.** Para cada producto se muestra la configuración que eligió el criterio AICc: en ETS, el tipo de error, de tendencia y de estacionalidad; en SARIMA, los órdenes (p, d, q) y (P, D, Q). Una configuración de ETS sin tendencia ni estacionalidad equivale a un suavizado exponencial cuyo parámetro se estima de los datos, y una diferenciación en SARIMA (d = 1) indica que el nivel de la serie cambia en el tiempo. Con historias cortas, el criterio tiende a preferir los modelos más simples, porque modelar la estacionalidad no compensa los parámetros adicionales.

#### 3.5 Aprendizaje profundo: LSTM e híbrido ARIMA-CNN-LSTM

- **LSTM:** red neuronal recurrente que recibe los últimos 6 meses y pronostica los 3 siguientes a la vez. La serie se divide por su promedio antes de entrenar.
- **Híbrido ARIMA-CNN-LSTM** (Huang y Zhou, 2025): el SARIMA del paso anterior captura la parte lineal de la serie. Una red con una capa convolucional (CNN) y una LSTM aprende el patrón de los **residuos** del SARIMA. El pronóstico final es el del SARIMA más la corrección de los residuos.

Las dos redes se reentrenan en cada origen con la historia disponible, igual que los demás métodos. Son redes pequeñas (16 neuronas en la LSTM, 150 pasadas sobre los datos) porque cada producto tiene solo 30 a 40 meses: con tan pocos datos, una red grande memorizaría la historia. Los resultados de estos dos métodos deben leerse con esa limitación en mente. Esta celda es la más lenta del notebook (algunos minutos).

In [ ]:
def ventanas(s):
    idx = range(VENTANA, len(s) - H_MAX + 1)
    X = np.array([s[i - VENTANA:i] for i in idx])[..., None]
    Y = np.array([s[i:i + H_MAX] for i in idx])
    return X, Y

def red_pronostico(serie, cnn, escala):
    escala = escala if escala > 0 else 1.0
    s = np.asarray(serie, float) / escala
    X, Y = ventanas(s)
    tf.keras.backend.clear_session(); tf.keras.utils.set_random_seed(SEED)
    capas = [tf.keras.Input(shape=(VENTANA, 1))]
    if cnn:
        capas.append(tf.keras.layers.Conv1D(8, 2, activation="relu"))
    capas += [tf.keras.layers.LSTM(16), tf.keras.layers.Dense(H_MAX)]
    red = tf.keras.Sequential(capas)
    red.compile(optimizer=tf.keras.optimizers.Adam(0.01), loss="mse")
    red.fit(np.tile(X, (REPETICIONES, 1, 1)), np.tile(Y, (REPETICIONES, 1)), epochs=EPOCAS, batch_size=8, verbose=0)
    return red.predict(s[-VENTANA:][None, :, None], verbose=0)[0] * escala

if not HAY_TF:
    print("Se omite esta sección: TensorFlow no está disponible (ver la celda de instalación al inicio).")
else:
    t0 = time.time(); filas_dl = []
    for k in PRODUCTOS:
        y = SERIES[k][COL_Y].to_numpy(float); n = len(y)
        for L in range(n - N_PRUEBA - H_MAX + 1, n):
            obs = y[:L]
            agregar(filas_dl, k, L, "LSTM", red_pronostico(obs, cnn=False, escala=obs.mean()))
            if (k, L) in ARIMA_FIT:
                pron_arima, resid = ARIMA_FIT[(k, L)]
                resid = resid[2:]                      # los primeros residuos dependen de la inicialización del SARIMA
                agregar(filas_dl, k, L, "Híbrido ARIMA-CNN-LSTM",
                        pron_arima + red_pronostico(resid, cnn=True, escala=resid.std()))
        print(f"  {k} listo ({time.time() - t0:.0f} s)")
    PRED = pd.concat([PRED, pd.DataFrame(filas_dl)], ignore_index=True)
    print(f"LSTM e híbrido: {len(filas_dl):,} pronósticos en {time.time() - t0:.0f} s")

### 4. Resultados

#### 4.1 Error por producto y método

In [ ]:
# escala del MASE: error de repetir el mes anterior, en el periodo de entrenamiento
ESCALA = {k: np.abs(np.diff(SERIES[k][COL_Y].to_numpy(float)[:-N_PRUEBA])).mean() for k in PRODUCTOS}

def metricas(g):
    err = g["pred"] - g["real"]
    return pd.Series({"MAE": err.abs().mean(),
                      "WMAPE": err.abs().sum() / g["real"].sum() if g["real"].sum() > 0 else np.nan,
                      "Sesgo": err.sum() / g["real"].sum() if g["real"].sum() > 0 else np.nan,
                      "MASE": err.abs().mean() / ESCALA[g.name[0]] if ESCALA[g.name[0]] > 0 else np.nan})

MET = PRED.groupby(["ItemCode", "h", "metodo"]).apply(metricas).reset_index()
SIMPLES = ["Ingenuo", "Estacional ingenuo", "Promedio móvil 3", "Suavizado exponencial"]
MODELOS = [m for m in ["ETS", "SARIMA", "XGBoost", "Random Forest", "LSTM", "Híbrido ARIMA-CNN-LSTM"] if m in set(PRED["metodo"])]
ORDEN = SIMPLES + MODELOS

for h in HORIZONTES:
    print(f"\nMASE a {h} mes(es) — menor es mejor")
    t = MET[MET["h"] == h].pivot(index="ItemCode", columns="metodo", values="MASE")[ORDEN]
    t["Mejor método"] = t.idxmin(axis=1)
    display(t.rename_axis("Producto").round(2))

#### 4.2 Resumen de los productos piloto

Para resumir los productos se usa la **mediana del MASE** (no la deja mover un solo producto con error muy grande) y el **WMAPE total**, que suma los errores y las unidades de todos los productos. También se cuenta en cuántos productos cada modelo le gana al **mejor método simple** de cada producto (el de menor error en la prueba: una referencia exigente, porque se elige mirando los resultados).

In [ ]:
filas = []
for h in HORIZONTES:
    m = MET[MET["h"] == h]
    mejor_simple = m[m["metodo"].isin(SIMPLES)].groupby("ItemCode")["MASE"].min()
    for nom in ORDEN:
        g = PRED[(PRED["h"] == h) & (PRED["metodo"] == nom)]
        mm = m[m["metodo"] == nom].set_index("ItemCode")["MASE"]
        filas.append({"Horizonte": h, "Método": nom, "MASE mediana": mm.median(),
                      "WMAPE total": (g["pred"] - g["real"]).abs().sum() / g["real"].sum(),
                      "Productos en que gana al mejor simple": int((mm < mejor_simple.reindex(mm.index)).sum()) if nom not in SIMPLES else None})
RESUMEN = pd.DataFrame(filas)
display(RESUMEN.pivot(index="Método", columns="Horizonte", values=["MASE mediana", "WMAPE total"]).loc[ORDEN].round(3))
print(f"Productos (de {len(PRODUCTOS)}) en que cada modelo le gana al mejor método simple")
display(RESUMEN[RESUMEN["Método"].isin(MODELOS)]
        .pivot(index="Método", columns="Horizonte", values="Productos en que gana al mejor simple").loc[MODELOS].astype(int))

**Cómo leer la comparación.** El MASE mediano resume el error típico por producto en una escala común: un valor menor que 1 indica que el método se equivoca menos que el pronóstico ingenuo. El WMAPE mide el error en unidades del conjunto, por lo que pesan más los productos de mayor volumen. El conteo de victorias indica en cuántos productos cada método supera al mejor método simple de ese producto, que es la referencia exigente del experimento. Un método conviene cuando es consistente entre productos y horizontes, no cuando gana en un solo caso. Los resultados obtenidos con los datos de la empresa y su interpretación se presentan en el documento de la tesis.

#### 4.3 Pronóstico frente a lo real

Para cada producto, a un mes: lo pedido (línea oscura) y el mejor método simple de ese producto (gris punteado), frente a dos métodos a la vez para que la gráfica se pueda leer. Una figura por familia: aprendizaje automático, estadísticos y aprendizaje profundo.

In [ ]:
def graficar(par, titulo, h=1):
    par = [m for m in par if m in MODELOS]
    if not par:
        print(f"{titulo}: no hay pronósticos de estos métodos (¿falta TensorFlow?)."); return
    fig, axes = plt.subplots(len(PRODUCTOS), 1, figsize=(13, 2.8 * len(PRODUCTOS)), squeeze=False)
    for ax, k in zip(axes[:, 0], PRODUCTOS):
        d = SERIES[k]; contexto = d.index[-(N_PRUEBA + 12):]
        ax.plot(contexto, d.loc[contexto, COL_Y], color=COLORES["Real"], lw=1.8, marker="o", ms=3, label="Real")
        m = MET[(MET["ItemCode"] == k) & (MET["h"] == h)].set_index("metodo")["MASE"]
        simple = m[SIMPLES].idxmin()
        for nom, estilo, color in [(p, "--", COLORES[p]) for p in par] + [(simple, ":", COLORES["Mejor método simple"])]:
            g = PRED[(PRED["ItemCode"] == k) & (PRED["h"] == h) & (PRED["metodo"] == nom)].sort_values("fecha")
            ax.plot(g["fecha"], g["pred"], ls=estilo, color=color, lw=1.8, marker="s", ms=3, label=f"{nom} (MASE {m[nom]:.2f})")
        ax.set_title(f"{k}", loc="left", fontsize=10)
        ax.set_ylim(bottom=0); ax.grid(alpha=.2); ax.legend(fontsize=8, frameon=False, loc="upper left", bbox_to_anchor=(1.01, 1))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%b-%y"))
    fig.suptitle(titulo, x=0.01, ha="left", fontweight="bold")
    plt.tight_layout(); plt.show()

graficar(["XGBoost", "Random Forest"], "Aprendizaje automático, pronóstico a 1 mes")
graficar(["ETS", "SARIMA"], "Métodos estadísticos, pronóstico a 1 mes")
graficar(["LSTM", "Híbrido ARIMA-CNN-LSTM"], "Aprendizaje profundo, pronóstico a 1 mes")

**Cómo leer las gráficas.** Cada gráfica compara la demanda real del periodo de prueba con los pronósticos de cada método. Conviene identificar en qué productos los métodos que siguen el último nivel (suavizado exponencial, ETS y SARIMA) reaccionan tarde ante un cambio de nivel, en cuáles los modelos que aprenden de los rezagos sobrerreaccionan a un pico y en cuáles los pronósticos oscilan más que la propia demanda, señal de que el modelo aprendió ruido en lugar de un patrón.

#### 4.4 Qué variables usan los modelos

Importancia de cada variable en el modelo entrenado con toda la historia de cada producto (horizonte de un mes), promediada entre los productos piloto. Si pesan más los rezagos y promedios, el modelo se apoya sobre todo en lo que pasó en los meses anteriores; si pesa el mes del año, aprovecha la estacionalidad.

In [ ]:
imp = {"XGBoost": [], "Random Forest": []}
for k in PRODUCTOS:
    tabla = construir_tabla(SERIES[k], 1)
    for nom, modelo in [("XGBoost", XGBRegressor(**PARAMS_XGB)), ("Random Forest", RandomForestRegressor(**PARAMS_RF))]:
        modelo.fit(tabla[VARIABLES], tabla["y"])
        v = pd.Series(modelo.feature_importances_, index=VARIABLES); imp[nom].append(v / v.sum())
IMP = pd.DataFrame({nom: pd.concat(v, axis=1).mean(axis=1) for nom, v in imp.items()}).sort_values("XGBoost")

fig, ax = plt.subplots(figsize=(8, 4.5)); yy = np.arange(len(IMP))
ax.barh(yy - .2, IMP["XGBoost"], height=.38, color=COLORES["XGBoost"], label="XGBoost")
ax.barh(yy + .2, IMP["Random Forest"], height=.38, color=COLORES["Random Forest"], label="Random Forest")
ax.set_yticks(yy); ax.set_yticklabels(IMP.index); ax.set_xlabel("Importancia relativa (suma = 1)")
ax.legend(frameon=False); ax.grid(axis="x", alpha=.2); plt.tight_layout(); plt.show()

### 5. Conclusiones de la comparación inicial

Esta sección cierra la primera comparación con dos decisiones de procedimiento:

1. **Modelo candidato.** Se toma como candidato el modelo de aprendizaje automático más consistente: el que más veces supera al mejor método simple de cada producto y tiene el menor MASE mediano en más horizontes.
2. **Limitaciones de la comparación.** Con pocos productos y pocos meses de prueba, las diferencias entre los mejores métodos pueden no ser significativas (se evalúa en la sección 6.5). El mejor método simple se elige mirando la prueba, lo que favorece a los métodos simples; los hiperparámetros se fijan de antemano (la sección 6.1 los ajusta); los meses con quiebre subestiman la demanda, y se supone que las variables comerciales del mes pronosticado se conocen de antemano.

Los resultados obtenidos con los datos de la empresa y su interpretación se presentan en el documento de la tesis.

**Siguiente paso.** Ajustar los hiperparámetros del candidato con validación temporal, entrenarlo con todos los productos de la línea para darle más datos y probar su combinación con el suavizado exponencial.

### 6. Siguiente paso: mejorar el modelo candidato (Random Forest)

Random Forest fue el modelo de aprendizaje automático más consistente. En esta sección se prueban tres mejoras, con la misma validación de origen móvil y los mismos meses de prueba que las secciones anteriores, para que los resultados sean comparables:

1. **Random Forest ajustado:** se eligen sus hiperparámetros con validación temporal **antes** del periodo de prueba.
2. **Random Forest global:** un solo modelo entrenado con todos los productos de la línea, para darle más datos que los 30 a 40 meses de un producto.
3. **Combinación con el suavizado exponencial:** el promedio de los pronósticos de Random Forest y del suavizado exponencial, el método simple con menor error en unidades.

#### 6.1 Ajuste de hiperparámetros con validación temporal

Se prueban 12 combinaciones de tres hiperparámetros: la profundidad máxima de los árboles (3, 5 o sin límite), el mínimo de meses por hoja (1 o 3) y la proporción de variables que mira cada división (todas o la mitad).

**Cómo se elige.** Para cada producto y horizonte se toman como validación los 6 meses anteriores al periodo de prueba, y cada uno se pronostica con origen móvil (entrenando solo con lo anterior), igual que en la prueba. Se elige la combinación con menor MASE promedio entre los 5 productos y los 3 horizontes. Así la elección no usa ningún dato de los meses de prueba.

In [ ]:

N_VAL = 6
GRID_RF = [dict(max_depth=md, min_samples_leaf=ml, max_features=mf)
           for md in (3, 5, None) for ml in (1, 3) for mf in (1.0, 0.5)]

def validar_rf(cfg):
    errores = []
    for k in PRODUCTOS:
        d = SERIES[k]; n = len(d)
        for h in HORIZONTES:
            tabla = construir_tabla(d, h)
            # meses de validación: ya conocidos en el primer origen de la prueba para este horizonte
            meses_val = d.index[n - N_PRUEBA - h - N_VAL + 1: n - N_PRUEBA - h + 1]
            ae = []
            for fecha in meses_val:
                prueba = tabla[tabla["fecha"] == fecha]
                if prueba.empty: continue
                entreno = tabla[tabla["fecha"] <= prueba["fecha_origen"].iloc[0]]
                if len(entreno) < 8: continue
                rf = RandomForestRegressor(n_estimators=200, random_state=SEED, n_jobs=-1, **cfg).fit(entreno[VARIABLES], entreno["y"])
                ae.append(abs(rf.predict(prueba[VARIABLES])[0] - prueba["y"].iloc[0]))
            if ae:
                errores.append(np.mean(ae) / ESCALA[k])
    return np.mean(errores)

t0 = time.time()
VALIDACION = pd.DataFrame([{**cfg, "MASE validación": validar_rf(cfg)} for cfg in GRID_RF])
VALIDACION["max_depth"] = VALIDACION["max_depth"].fillna("sin límite")
VALIDACION = VALIDACION.sort_values("MASE validación").reset_index(drop=True)
MEJOR_RF = {k: (None if v == "sin límite" else v) for k, v in VALIDACION.iloc[0][["max_depth", "min_samples_leaf", "max_features"]].items()}
MEJOR_RF["min_samples_leaf"] = int(MEJOR_RF["min_samples_leaf"]); MEJOR_RF["max_features"] = float(MEJOR_RF["max_features"])
if MEJOR_RF["max_depth"] is not None: MEJOR_RF["max_depth"] = int(MEJOR_RF["max_depth"])
print(f"Validación de {len(GRID_RF)} combinaciones en {time.time() - t0:.0f} s")
print(f"Combinación elegida: {MEJOR_RF} (la original era max_depth=4, min_samples_leaf=2, max_features=1.0)")
display(VALIDACION.round(3))


In [ ]:

# Random Forest ajustado: misma prueba de origen móvil de la sección 3.3, con la combinación elegida
filas = []
for k in PRODUCTOS:
    d = SERIES[k]
    for h in HORIZONTES:
        tabla = construir_tabla(d, h)
        for fecha in d.index[-N_PRUEBA:]:
            prueba = tabla[tabla["fecha"] == fecha]
            if prueba.empty: continue
            entreno = tabla[tabla["fecha"] <= prueba["fecha_origen"].iloc[0]]
            rf = RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=-1, **MEJOR_RF).fit(entreno[VARIABLES], entreno["y"])
            filas.append(dict(ItemCode=k, h=h, fecha=fecha, real=prueba["y"].iloc[0],
                              pred=max(float(rf.predict(prueba[VARIABLES])[0]), 0), metodo="Random Forest ajustado"))
PRED = pd.concat([PRED[PRED["metodo"] != "Random Forest ajustado"], pd.DataFrame(filas)], ignore_index=True)
print(f"Random Forest ajustado: {len(filas)} pronósticos")


#### 6.2 Random Forest global: un modelo con todos los productos de la línea

En lugar de un modelo por producto, se entrena **un solo Random Forest con las filas de todos los productos de la línea** y con él se pronostican los 5 piloto. Cada producto aporta sus propios meses, así que el modelo aprende de muchas más situaciones (subidas, caídas, quiebres) que las que tiene un solo producto.

Para que productos de tamaños muy distintos se puedan mezclar, las unidades de cada fila (rezagos, promedios y valor a pronosticar) se dividen por el promedio de los 12 meses anteriores al origen de ese producto, y el pronóstico se devuelve a unidades multiplicando por ese mismo valor. Se agrega como variable el tamaño del producto (logaritmo de ese promedio). La regla contra la fuga de información es la misma: en cada origen solo se usan filas cuyo mes objetivo ya ocurrió.

In [ ]:

PARAMS_RF_GLOBAL = dict(n_estimators=300, max_depth=6, min_samples_leaf=3, max_features=0.6, random_state=SEED, n_jobs=-1)

mes_linea = spark.table(TABLA_MES).toPandas()
mes_linea["ItemCode"] = mes_linea["ItemCode"].astype(str)
mes_linea = mes_linea[mes_linea["ItemCode"].isin(cls["ItemCode"])].copy()
mes_linea["ItemCode"] = mes_linea["ItemCode"].replace(ALIAS)      # los piloto con su alias; los demás solo se usan para entrenar
mes_linea["mes"] = pd.to_datetime(mes_linea["mes"]).dt.to_period("M").dt.to_timestamp()
SERIES_LINEA = {k: preparar_serie(g) for k, g in mes_linea.groupby("ItemCode")}
SERIES_LINEA = {k: d for k, d in SERIES_LINEA.items() if len(d) >= 12}
# Control de trazabilidad: la serie de cada piloto debe ser la misma en la ABT y en el dataset de la A2.4
coinciden = all(k in SERIES_LINEA and SERIES_LINEA[k].index.equals(SERIES[k].index)
                and np.allclose(SERIES_LINEA[k][COL_Y].to_numpy(float), SERIES[k][COL_Y].to_numpy(float))
                for k in PRODUCTOS)
print(f"Series del piloto iguales en {TABLA_PILOTO} y {TABLA_MES}: {coinciden}")

ESCALADAS = [f"lag{L}" for L in LAGS] + ["mm3", "mm6"]
VARIABLES_G = VARIABLES + ["log_escala"]

def tabla_global(k, h):
    d = SERIES_LINEA[k]
    t = construir_tabla(d, h)
    if t.empty: return t
    nivel = d[COL_Y].rolling(12, min_periods=1).mean().clip(lower=1)     # promedio de los 12 meses hasta el origen
    t["escala"] = t["fecha_origen"].map(nivel).to_numpy()
    for c in ESCALADAS + ["y"]:
        t[c + "_esc"] = t[c] / t["escala"]
    t["log_escala"] = np.log1p(t["escala"])
    t["ItemCode"] = k
    return t

t0 = time.time(); filas = []
for h in HORIZONTES:
    T = pd.concat([tabla_global(k, h) for k in SERIES_LINEA], ignore_index=True)
    X = lambda df: df[[c + "_esc" if c in ESCALADAS else c for c in VARIABLES_G]].to_numpy(float)
    for fecha in sorted({f for k in PRODUCTOS for f in SERIES[k].index[-N_PRUEBA:]}):
        prueba = T[(T["fecha"] == fecha) & T["ItemCode"].isin([k for k in PRODUCTOS if fecha in SERIES[k].index[-N_PRUEBA:]])]
        if prueba.empty: continue
        entreno = T[T["fecha"] <= prueba["fecha_origen"].iloc[0]]
        rf = RandomForestRegressor(**PARAMS_RF_GLOBAL).fit(X(entreno), entreno["y_esc"])
        pred = np.clip(rf.predict(X(prueba)), 0, None) * prueba["escala"].to_numpy()
        for (_, r), p in zip(prueba.iterrows(), pred):
            filas.append(dict(ItemCode=r["ItemCode"], h=h, fecha=fecha, real=r["y"], pred=float(p), metodo="Random Forest global"))
PRED = pd.concat([PRED[PRED["metodo"] != "Random Forest global"], pd.DataFrame(filas)], ignore_index=True)
print(f"Productos de la línea usados para entrenar: {len(SERIES_LINEA)}")
print(f"Random Forest global: {len(filas)} pronósticos en {time.time() - t0:.0f} s")


#### 6.3 Combinación con el suavizado exponencial

Combinar pronósticos de métodos distintos suele reducir el error, porque sus equivocaciones no coinciden del todo. Se prueba el promedio simple de cada Random Forest con el suavizado exponencial. Es un promedio fijo (50 % y 50 %), así que tampoco usa información de la prueba.

In [ ]:

ancho = PRED.pivot_table(index=["ItemCode", "h", "fecha"], columns="metodo", values="pred")
reales = PRED.groupby(["ItemCode", "h", "fecha"])["real"].first()
combinaciones = {"Combinación RF ajustado + suavizado": "Random Forest ajustado",
                 "Combinación RF global + suavizado": "Random Forest global"}
nuevas = []
for nombre_c, rf_nom in combinaciones.items():
    c = ((ancho[rf_nom] + ancho["Suavizado exponencial"]) / 2).dropna().rename("pred").reset_index()
    c = c.merge(reales.reset_index(), on=["ItemCode", "h", "fecha"], how="left")
    c["metodo"] = nombre_c
    nuevas.append(c)
PRED = pd.concat([PRED[~PRED["metodo"].isin(combinaciones)]] + nuevas, ignore_index=True)
print(f"Combinaciones: {sum(len(c) for c in nuevas)} pronósticos")


#### 6.4 Resultados de las mejoras

Se comparan las versiones de Random Forest y las combinaciones con el suavizado exponencial (el mejor método simple en unidades) y con el Random Forest original. Las métricas son las mismas de la sección 4.

In [ ]:

MET = PRED.groupby(["ItemCode", "h", "metodo"]).apply(metricas).reset_index()
MEJORAS = ["Suavizado exponencial", "Random Forest", "Random Forest ajustado", "Random Forest global",
           "Combinación RF ajustado + suavizado", "Combinación RF global + suavizado"]

filas = []
for h in HORIZONTES:
    m = MET[MET["h"] == h]
    mejor_simple = m[m["metodo"].isin(SIMPLES)].groupby("ItemCode")["MASE"].min()
    for nom in MEJORAS:
        g = PRED[(PRED["h"] == h) & (PRED["metodo"] == nom)]
        mm = m[m["metodo"] == nom].set_index("ItemCode")["MASE"]
        filas.append({"Horizonte": h, "Método": nom, "MASE mediana": mm.median(),
                      "WMAPE total": (g["pred"] - g["real"]).abs().sum() / g["real"].sum(),
                      "Gana al mejor simple (de 5)": int((mm < mejor_simple.reindex(mm.index)).sum()) if nom not in SIMPLES else "—"})
RES_MEJORAS = pd.DataFrame(filas)
display(RES_MEJORAS.pivot(index="Método", columns="Horizonte", values=["MASE mediana", "WMAPE total"]).loc[MEJORAS].round(3))
print("Productos en que cada método le gana al mejor método simple de ese producto")
display(RES_MEJORAS.pivot(index="Método", columns="Horizonte", values="Gana al mejor simple (de 5)").loc[MEJORAS])

print("\nMASE por producto a 1 mes")
display(MET[(MET["h"] == 1) & MET["metodo"].isin(MEJORAS)].pivot(index="ItemCode", columns="metodo", values="MASE")[MEJORAS].rename_axis("Producto").round(2))


In [ ]:

COLORES.update({"Random Forest ajustado": "#2a78d6", "Random Forest global": "#173f80",
                "Combinación RF ajustado + suavizado": "#1baf7a", "Combinación RF global + suavizado": "#0f766e"})
MODELOS = list(dict.fromkeys(MODELOS + MEJORAS[2:]))
graficar(["Random Forest global", "Combinación RF global + suavizado"], "Random Forest global y su combinación, pronóstico a 1 mes")
graficar(["Random Forest ajustado", "Combinación RF ajustado + suavizado"], "Random Forest ajustado y su combinación, pronóstico a 1 mes")


**Cómo leer los resultados.** La primera tabla muestra la combinación de hiperparámetros elegida y el error de validación de cada combinación: si todas quedan cerca, el modelo es poco sensible a sus hiperparámetros. La segunda compara las variantes de Random Forest (ajustado, global y combinado con el suavizado exponencial) con el modelo original y con el mejor método simple de cada producto, en MASE mediano, WMAPE y número de victorias por horizonte. Que el modelo global mejore al modelo por producto indica que darle más series de las cuales aprender compensa la historia corta de cada una. Los resultados obtenidos con los datos de la empresa y su interpretación se presentan en el documento de la tesis.

#### 6.5 ¿Las diferencias son estadísticamente significativas?

Con 5 productos y 9 meses de prueba, una diferencia pequeña en el MASE mediano puede deberse al azar. Para comprobarlo se compara cada variante con el suavizado exponencial **mes a mes y producto a producto**: para cada uno de los 45 pronósticos de cada horizonte se calcula el error absoluto de los dos métodos, dividido por la escala del MASE del producto para que los productos de mayor volumen no dominen, y se aplica la prueba de rangos con signo de Wilcoxon a esos pares.

- `Meses con menor error`: en cuántos de los 45 pronósticos la variante se equivocó menos que el suavizado exponencial.
- `Diferencia mediana (MASE)`: mediana de (error de la variante − error del suavizado); negativa significa que la variante se equivoca menos.
- `p (Wilcoxon)`: menor a 0,05 indica una diferencia estadísticamente significativa.

Los errores de meses seguidos de un mismo producto no son del todo independientes, así que el valor p debe leerse como una guía y no como una prueba exacta.

In [ ]:
from scipy.stats import wilcoxon
REF = "Suavizado exponencial"

def errores_escalados(nom, h):
    g = PRED[(PRED["h"] == h) & (PRED["metodo"] == nom)].set_index(["ItemCode", "fecha"])
    escala = g.index.get_level_values("ItemCode").map(ESCALA).to_numpy(float)
    return ((g["pred"] - g["real"]).abs() / escala).rename(nom)

filas = []
for h in HORIZONTES:
    e_ref = errores_escalados(REF, h)
    for nom in MEJORAS[1:]:
        par = pd.concat([errores_escalados(nom, h), e_ref], axis=1).dropna()
        dif = par[nom] - par[REF]
        p = wilcoxon(par[nom], par[REF]).pvalue if (dif != 0).any() else np.nan
        filas.append({"Horizonte": h, "Método": nom, "Pares": len(par), "Meses con menor error": int((dif < 0).sum()),
                      "Diferencia mediana (MASE)": round(float(dif.median()), 3), "p (Wilcoxon)": round(float(p), 3)})
PRUEBA = pd.DataFrame(filas)
display(PRUEBA.pivot(index="Método", columns="Horizonte",
                     values=["Meses con menor error", "Diferencia mediana (MASE)", "p (Wilcoxon)"]).loc[MEJORAS[1:]])

**Cómo leer la prueba.** Para cada variante y horizonte, la tabla muestra en cuántos de los pronósticos se equivoca menos que el suavizado exponencial, la diferencia mediana de los errores escalados y el valor p de la prueba de Wilcoxon. Un valor p mayor que 0,05 indica que la diferencia puede deberse al azar: con pocos productos y pocos meses de prueba, una ventaja que va en la misma dirección en todos los horizontes no basta para afirmar que una variante es mejor. Los resultados obtenidos con los datos de la empresa y su interpretación se presentan en el documento de la tesis.

#### 6.6 Modelo propuesto para los productos piloto

**Modelo propuesto: combinación del Random Forest ajustado con el suavizado exponencial** (promedio simple de los dos pronósticos).

Se elige con tres criterios, en este orden:

1. **Consistencia entre productos y horizontes.** Es la variante que más veces supera al mejor método simple de cada producto (número de combinaciones de producto y horizonte en que lo supera).
2. **Error en unidades a un mes**, el horizonte más cercano a las decisiones de reposición: su WMAPE debe estar entre los más bajos del experimento.
3. **Robustez.** Al promediar un modelo que aprende de las variables (rezagos, mes del año, descuento) con uno que sigue el nivel reciente, cada uno compensa errores del otro: Random Forest reacciona mejor a los cambios de nivel y el suavizado exponencial evita sobrerreaccionar a picos.

**Qué significa la prueba de la sección 6.5.** La combinación no es significativamente mejor que el suavizado exponencial. Se propone porque es la más consistente y porque, al incluir el suavizado exponencial, no puede alejarse mucho de él: en el peor caso rinde como el método simple, y en los productos con cambios de nivel aporta lo que aprende Random Forest. Si se prefiere el modelo más simple con un error equivalente, el suavizado exponencial es una alternativa válida y defendible.

**Advertencia metodológica.** La elección se hace entre variantes definidas antes de la prueba, pero comparándolas en los mismos meses de prueba. Por eso el error del modelo propuesto puede ser algo optimista. La forma de confirmarlo es evaluarlo en los meses siguientes, a medida que llegan nuevos datos, sin volver a ajustar nada.

### 7. Modelo final y pronóstico de los próximos tres meses

Con el modelo ya elegido (sección 6.6), esta sección hace lo que pide el objetivo: **entrenar el modelo con toda la historia, guardar sus archivos (Actividad 3.2) y pronosticar los tres meses siguientes al último dato (Actividad 3.4)**.

1. **Entrenamiento final.** Para cada producto y horizonte se entrena el Random Forest ajustado (los hiperparámetros elegidos en la sección 6.1) con todos los meses disponibles, y se calcula el suavizado exponencial de toda la serie. El pronóstico es el promedio de los dos, igual que en la prueba.
2. **Archivos del modelo.** Cada Random Forest se guarda en la carpeta `Files/modelos/fase3` del Lakehouse, junto con un archivo `modelo_propuesto.json` que describe el modelo: variables, hiperparámetros, pesos de la combinación, parámetro del suavizado y último mes de entrenamiento. Los archivos usan los alias de los productos, no sus códigos.
3. **Variables comerciales del futuro.** El modelo usa la promoción y el descuento del mes pronosticado. Como aún no se conocen, se supone **un mes sin promoción**. Si el área comercial ya tiene planeada una promoción, se registra en `PROMO_FUTURA` y el pronóstico la tiene en cuenta.
4. **Error típico.** Junto al pronóstico se muestra el error absoluto medio que tuvo el modelo con ese producto y en ese horizonte durante los nueve meses de prueba, y el rango que resulta de sumarlo y restarlo al pronóstico. No es un intervalo estadístico formal, pero indica cuánto suele desviarse la demanda real de lo pronosticado en cada producto.

In [ ]:
import os, json, joblib

# ------------------------------------------------------------------ parámetros de la sección 7
CARPETA_MODELOS = "/lakehouse/default/Files/modelos/fase3"     # carpeta Files del Lakehouse por defecto
PROMO_FUTURA = {}       # opcional, ej.: {("Producto 1", "2026-10"): {"mes_promocional": 1, "descuento_incremental": 5.0}}
MODELO = "Combinación RF ajustado + suavizado"

if not os.path.isdir("/lakehouse/default/Files"):
    CARPETA_MODELOS = "modelos_fase3"
    print(f"Aviso: no hay un Lakehouse por defecto montado; los archivos se guardan en la carpeta local {CARPETA_MODELOS}")
os.makedirs(CARPETA_MODELOS, exist_ok=True)

def fila_futura(k, d, mes_obj):
    """Variables del mes que se pronostica, con la información disponible al último mes de la serie."""
    y = d[COL_Y].to_numpy(float); q = d["pct_dias_quiebre"].to_numpy(float); t = len(y)
    f = {f"lag{L}": y[t - L] for L in LAGS}
    f["mm3"], f["mm6"] = y[t - 3:t].mean(), y[t - 6:t].mean()
    f["quiebre_ultimo_mes"] = q[t - 1]
    f["mes_sin"], f["mes_cos"] = np.sin(2 * np.pi * mes_obj.month / 12), np.cos(2 * np.pi * mes_obj.month / 12)
    plan = PROMO_FUTURA.get((k, f"{mes_obj:%Y-%m}"), {})
    for c in COMERCIALES:
        f[c] = float(plan.get(c, 0.0))                 # sin plan comercial: mes sin promoción
    return pd.DataFrame([f])[VARIABLES]

# Error típico: error absoluto medio del modelo propuesto en los 9 meses de prueba, por producto y horizonte
g = PRED[PRED["metodo"] == MODELO]
ERROR_TIPICO = (g["pred"] - g["real"]).abs().groupby([g["ItemCode"], g["h"]]).mean()

filas, archivos = [], []
for k in PRODUCTOS:
    d = SERIES[k]; y = d[COL_Y].to_numpy(float)
    nivel_ses = float(suavizado(y))
    for h in HORIZONTES:
        tabla = construir_tabla(d, h)
        rf = RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=-1, **MEJOR_RF).fit(tabla[VARIABLES], tabla["y"])
        mes_obj = d.index[-1] + pd.DateOffset(months=h)
        p_rf = max(float(rf.predict(fila_futura(k, d, mes_obj))[0]), 0.0)
        p = 0.5 * p_rf + 0.5 * nivel_ses
        e = float(ERROR_TIPICO.get((k, h), np.nan))
        filas.append({"Producto": k, "Último mes con datos": f"{d.index[-1]:%Y-%m}", "Horizonte (meses)": h,
                      "Mes pronosticado": f"{mes_obj:%Y-%m}", "Random Forest ajustado": round(p_rf),
                      "Suavizado exponencial": round(nivel_ses), "Pronóstico (modelo propuesto)": round(p),
                      "Error típico en la prueba (±)": round(e), "Rango típico inferior": round(max(p - e, 0)),
                      "Rango típico superior": round(p + e)})
        nombre = f"rf_ajustado_{k.replace(' ', '_')}_h{h}.joblib"
        joblib.dump({"modelo": rf, "variables": VARIABLES, "producto": k, "horizonte": h,
                     "entrenado_hasta": f"{d.index[-1]:%Y-%m}", "hiperparametros": MEJOR_RF},
                    os.path.join(CARPETA_MODELOS, nombre))
        archivos.append(nombre)

FUTURO = pd.DataFrame(filas)
descripcion = {
    "modelo": "Combinación 50/50 de Random Forest ajustado y suavizado exponencial simple",
    "variable_objetivo": COL_Y, "horizontes": list(HORIZONTES), "estrategia": "directa: un Random Forest por producto y horizonte",
    "pesos": {"Random Forest ajustado": 0.5, "Suavizado exponencial": 0.5}, "alpha_suavizado": 0.3,
    "hiperparametros_rf": {**MEJOR_RF, "n_estimators": 300, "random_state": SEED}, "variables": VARIABLES,
    "supuesto_comercial": "mes sin promoción salvo lo indicado en PROMO_FUTURA",
    "nivel_suavizado_final": {k: round(float(suavizado(SERIES[k][COL_Y].to_numpy(float))), 1) for k in PRODUCTOS},
    "entrenado_hasta": {k: f"{SERIES[k].index[-1]:%Y-%m}" for k in PRODUCTOS},
    "archivos": archivos}
with open(os.path.join(CARPETA_MODELOS, "modelo_propuesto.json"), "w", encoding="utf-8") as fh:
    json.dump(descripcion, fh, ensure_ascii=False, indent=2, default=str)

print(f"Archivos del modelo guardados en {CARPETA_MODELOS}: {len(archivos)} Random Forest + modelo_propuesto.json")
display(FUTURO)

In [ ]:
fig, axes = plt.subplots(len(PRODUCTOS), 1, figsize=(12, 2.6 * len(PRODUCTOS)), sharex=True)
color_p = COLORES.get(MODELO, "#2a78d6")
for ax, k in zip(np.atleast_1d(axes), PRODUCTOS):
    d = SERIES[k][COL_Y].iloc[-24:]
    f = FUTURO[FUTURO["Producto"] == k]
    fechas = pd.to_datetime(f["Mes pronosticado"])
    ax.plot(d.index, d.values, color=COLORES["Real"], marker="o", ms=3, lw=1.6, label="Real")
    ax.plot([d.index[-1], *fechas], [d.values[-1], *f["Pronóstico (modelo propuesto)"]], color=color_p, ls="--", lw=1.2)
    ax.errorbar(fechas, f["Pronóstico (modelo propuesto)"],
                yerr=[f["Pronóstico (modelo propuesto)"] - f["Rango típico inferior"],
                      f["Rango típico superior"] - f["Pronóstico (modelo propuesto)"]],
                fmt="o", color=color_p, ms=6, capsize=4, lw=1.2, label="Pronóstico ± error típico de la prueba")
    ax.set_title(k, loc="left", fontsize=10); ax.set_ylabel("Unidades"); ax.grid(alpha=.25)
np.atleast_1d(axes)[0].legend(loc="upper left", fontsize=8, frameon=False)
fig.suptitle("Pronóstico de los próximos tres meses con el modelo propuesto", y=1.0)
plt.tight_layout(); plt.show()

**Cómo usar este pronóstico.** La columna *Pronóstico (modelo propuesto)* es la cantidad esperada de unidades pedidas en cada mes. Para la compra, el error típico indica cuánto suele desviarse la demanda en cada producto: el límite superior del rango sirve de referencia para el inventario de seguridad en los productos donde un agotado cuesta más. El pronóstico debe recalcularse cada cierre de mes, con el mes nuevo incluido, ejecutando de nuevo el notebook; los archivos de la carpeta `modelos/fase3` se sobrescriben con el modelo actualizado.

In [ ]:
import re
def guardar(pdf, nombre_tabla):
    # Delta no acepta espacios, comas, punto y coma, llaves, paréntesis ni '=' en los nombres de columna
    pdf = pdf.rename(columns=lambda c: re.sub(r"[ ,;{}()\n\t=]+", "_", str(c)).strip("_"))
    if pdf.empty:
        print(f"{nombre_tabla}: sin filas, no se guarda"); return
    spark.createDataFrame(pdf).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(nombre_tabla)
    print(f"Guardada {nombre_tabla}: {len(pdf):,} filas")

guardar(RESUMEN_SERIES, "fs3_piloto_series")
guardar(PRED, "fs3_piloto_predicciones")
guardar(MET, "fs3_piloto_metricas")
guardar(PICOS, "fs3_piloto_picos")
guardar(FUTURO, "fs3_piloto_pronostico_futuro")
# Equivalencia alias -> código real: solo queda en el Lakehouse, para trazabilidad interna; no se muestra en el notebook
guardar(pd.DataFrame({"Producto": list(ALIAS.values()), "ItemCode": list(ALIAS.keys())}), "fs3_piloto_alias")